In [1]:
import numpy as np
import pandas as pd
import os, random, gc
import warnings

warnings.filterwarnings("ignore")

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    MaxAbsScaler,
    RobustScaler,
)
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform as sp_randFloat
from scipy.stats import randint as sp_randInt

from catboost import CatBoostRegressor
from xgboost import XGBClassifier

TRAIN_PATH = "../input/tabular-playground-series-may-2022/train.csv"
TEST_PATH = "../input/tabular-playground-series-may-2022/test.csv"
SAMPLE_SUBMISSION_PATH = (
    "../input/tabular-playground-series-may-2022/sample_submission.csv"
)
SUBMISSION_PATH = "submission.csv"

ID = "id"
TARGET = "target"

SEED = 2022


def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything()




In [2]:
def reduce_memory_usage(df, verbose=True):
    numerics = ["int8", "int16", "int32", "int64", "float16", "float32", "float64"]
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if (
                    c_min > np.finfo(np.float16).min
                    and c_max < np.finfo(np.float16).max
                ):
                    df[col] = df[col].astype(np.float16)
                elif (
                    c_min > np.finfo(np.float32).min
                    and c_max < np.finfo(np.float32).max
                ):
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(
            "Mem. usage decreased to {:.2f} Mb ({:.1f}% reduction)".format(
                end_mem, 100 * (start_mem - end_mem) / start_mem
            )
        )
    return df




In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)



In [4]:
train = reduce_memory_usage(train)
# Keep the original sampling behavior (core training approach) to match the provided solution flow.
train = train.sample(10000, random_state=SEED)
gc.collect()
test = reduce_memory_usage(test)
gc.collect()



Mem. usage decreased to 45.01 Mb (77.7% reduction)
Mem. usage decreased to 5.53 Mb (77.3% reduction)


0

In [5]:
train.describe(include="O")



,f_27
count,10000
unique,9974
top,ADBABACCDC
freq,2


In [6]:
train["f_27"]



505211    ABBAAAAECD
601201    ADBEBABNEB
677100    BAAACABODD
105636    BEAABAAIAC
207352    BBBBCBDSAA
             ...    
58504     AAADAAIHAA
620179    BEBDCABRAA
392959    BAAEDABIDF
576239    ABAIABFACB
694529    BCACCAACAA
Name: f_27, Length: 10000, dtype: object

In [7]:
# Bugfix: ensure both train/test drop the same non-numeric categorical feature.
y = train[TARGET].astype(np.int8)
X = train.drop([ID, TARGET, "f_27"], axis=1)
X_test = test.drop([ID, "f_27"], axis=1)



In [8]:
# Bugfix: Kaggle CPU environment often has no GPU; use CPU histogram.
# Score/metric alignment: use classifier with probabilistic output for ROC-AUC.
MODEL_TREE_METHOD = "hist"

model = XGBClassifier(
    tree_method=MODEL_TREE_METHOD,
    objective="binary:logistic",
    random_state=SEED,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=-1,
    eval_metric="auc",
)
model.fit(X, y)



XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=-1,
              num_parallel_tree=None, random_state=2022, ...)

In [9]:
sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)

# Predict probabilities for the positive class.
pred = model.predict_proba(X_test)[:, 1].astype(np.float32)

# Ensure alignment with sample submission ids (defensive, score-neutral).
test_ids = test[ID].values
pred_series = pd.Series(pred, index=test_ids)
sub[TARGET] = sub[ID].map(pred_series).astype(np.float32)

# If any ids are missing (shouldn't happen), fill with mean prediction.
if sub[TARGET].isna().any():
    sub[TARGET] = sub[TARGET].fillna(float(np.nanmean(pred)))

sub.to_csv(SUBMISSION_PATH, index=False)
sub.head()

,id,target
0,800000,0.644719
1,800001,0.593013
2,800002,0.987077
3,800003,0.528048
4,800004,0.257203
